# EDUAI · Chatterbox Multilingual V3 (Kaggle GPU T4)

Notebook privado, basado en el procedimiento **ya ejecutado con éxito**: Python 3.11 aislado, CUDA 12.6, Chatterbox V3, Perth real y salida WAV/ZIP.

1. **Settings → Accelerator → GPU T4 x2**, **Turn on internet**.
2. Deja este notebook **Private**: vas a subir una referencia de tu voz. Solo voces propias o autorizadas con consentimiento.
3. Ejecuta todas las celdas en orden. La instalación inicial descarga bibliotecas y pesos desde PyPI, GitHub oficial y ModelScope; no se utilizan servicios de Hugging Face.
4. Cambia `TEXTO` cuando quieras escuchar tu propia voz. Genera y descarga el ZIP desde **Output**.
5. Abre EDUAI → Mis voces y pulsa **Importar ZIP o WAV de Kaggle** en el perfil autorizado. El resultado quedará en tu biblioteca privada.
6. Para otra sesión, abre este notebook desde el botón guardado en EDUAI, vuelve a activar GPU e Internet y ejecuta las celdas.

**Limitación:** Kaggle no es un servidor GPU permanente; cada sesión requiere iniciarlo. La voz asociada en EDUAI es una **referencia zero-shot**, no un modelo entrenado individual. No introduzcas tokens de EDUAI, Supabase ni Google en Kaggle.

In [ ]:
import sys, subprocess
from pathlib import Path

subprocess.run([sys.executable,"-m","pip","install","-q","uv"],check=True)
UV=[sys.executable,"-m","uv"]
ENV_DIR=Path("/kaggle/working/eduai_voice_env")
PY=str(ENV_DIR/"bin"/"python")
subprocess.run(UV+["venv","--python","3.11",str(ENV_DIR)],check=True)
print("Python EDUAI:", PY)

In [ ]:
import os, subprocess

env=os.environ.copy()
env.update(HF_HUB_OFFLINE="1",TRANSFORMERS_OFFLINE="1",UV_HTTP_TIMEOUT="300")

def instalar(*args):
    subprocess.run(UV+["pip","install","--python",PY,*args],check=True,env=env)

instalar("torch==2.6.0","torchvision==0.21.0","torchaudio==2.6.0",
        "--index-url","https://download.pytorch.org/whl/cu126")
instalar("chatterbox-tts==0.1.7","modelscope>=1.20,<3")
instalar("--reinstall","--no-deps",
        "git+https://github.com/resemble-ai/chatterbox.git@5de7a54aa4e5e2baadb0182dde554908b48b85c2")
instalar("setuptools<81")  # El watermarker Perth aún importa pkg_resources
print("Bibliotecas instaladas en el entorno aislado")

In [ ]:
prueba = """
import torch, torchaudio, torchvision, inspect, perth
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
assert torch.cuda.is_available(), "GPU T4 no disponible"
assert "t3_model" in inspect.signature(ChatterboxMultilingualTTS.from_local).parameters
assert perth.PerthImplicitWatermarker is not None, "Perth no importado"
watermarker=perth.PerthImplicitWatermarker()
print("Torch:",torch.__version__)
print("Torchaudio:",torchaudio.__version__)
print("Torchvision:",torchvision.__version__)
print("CUDA activa:",torch.cuda.is_available())
print("PERTH FUNCIONANDO")
print("CHATTERBOX V3 LISTO")
"""
subprocess.run([PY,"-c",prueba],check=True,env=env)

In [ ]:
MODEL_DIR=Path("/kaggle/working/eduai_model_v3")
MODEL_DIR.mkdir(parents=True,exist_ok=True)
model_files=["ve.pt","s3gen.pt","t3_mtl23ls_v3.safetensors",
             "grapheme_mtl_merged_expanded_v1.json"]
faltan=[name for name in model_files if not (MODEL_DIR/name).is_file()]
if faltan:
    subprocess.run([str(Path(PY).parent/"modelscope"),"download",
        "--model","ResembleAI/chatterbox",*faltan,"--local_dir",str(MODEL_DIR)],check=True)
for name in model_files:
    file=MODEL_DIR/name
    assert file.is_file() and file.stat().st_size>2048, "Modelo incompleto: "+name
print("MODELO V3 DESCARGADO CORRECTAMENTE")

In [ ]:
import ipywidgets as widgets
from IPython.display import display
subir_voz=widgets.FileUpload(
    accept=".wav,.mp3,.m4a,.ogg",
    multiple=False,
    description="Subir mi voz"
)
display(subir_voz)
print("Sube una muestra propia de 10 a 30 segundos y ejecuta la celda siguiente.")

In [ ]:
import tempfile, wave
archivos=subir_voz.value
if not archivos:
    raise ValueError("Primero debes subir tu referencia vocal")
item=next(iter(archivos.values())) if isinstance(archivos,dict) else archivos[0]
contenido=bytes(item["content"])
if len(contenido)>12*1024*1024:
    raise ValueError("La muestra supera 12 MB")
ext=Path(item.get("name","referencia.wav")).suffix or ".wav"
with tempfile.NamedTemporaryFile(suffix=ext,delete=False) as tmp:
    tmp.write(contenido)
    entrada=Path(tmp.name)
REFERENCIA=Path("/tmp/eduai_referencia.wav")
try:
    subprocess.run(["ffmpeg","-y","-hide_banner","-loglevel","error",
        "-i",str(entrada),"-ac","1","-ar","24000",
        "-acodec","pcm_s16le",str(REFERENCIA)],check=True)
finally:
    entrada.unlink(missing_ok=True)
with wave.open(str(REFERENCIA),"rb") as recording:
    duration=recording.getnframes()/recording.getframerate()
if not 10 <= duration <= 30:
    REFERENCIA.unlink(missing_ok=True)
    raise ValueError("La muestra debe durar entre 10 y 30 segundos")
print("Muestra de voz preparada:",round(duration,1),"segundos")

In [ ]:
import os
from IPython.display import Audio, FileLink

TEXTO="Hola. Esta es una prueba de mi voz con Chatterbox V3 en EDUAI."
if not 1<=len(TEXTO.strip())<=550:
    raise ValueError("Escribe de 1 a 550 caracteres")
SALIDA=Path("/kaggle/working/eduai_voz_kaggle.wav")
script=Path("/tmp/eduai_generar_voz.py")
script.write_text("""
import os
import torch, torchaudio
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
modelo=ChatterboxMultilingualTTS.from_local(
    ckpt_dir=os.environ["EDUAI_MODEL_DIR"],device="cuda",t3_model="v3")
with torch.inference_mode():
    voz=modelo.generate(os.environ["EDUAI_TEXTO"],language_id="es",
                        audio_prompt_path=os.environ["EDUAI_REFERENCIA"])
torchaudio.save(os.environ["EDUAI_SALIDA"],voz.detach().cpu(),
                modelo.sr,format="wav",encoding="PCM_S",bits_per_sample=16)
print("VOZ GENERADA CORRECTAMENTE")
""",encoding="utf-8")
variables=env.copy()
variables.update(EDUAI_MODEL_DIR=str(MODEL_DIR),EDUAI_TEXTO=TEXTO.strip(),
                 EDUAI_REFERENCIA=str(REFERENCIA),EDUAI_SALIDA=str(SALIDA))
subprocess.run([PY,str(script)],check=True,env=variables)
display(Audio(filename=str(SALIDA)))
display(FileLink(str(SALIDA)))

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
ZIP=Path("/kaggle/working/eduai_mi_voz.zip")
assert SALIDA.is_file(), "Primero ejecuta la generación"
with ZipFile(ZIP,"w",ZIP_DEFLATED) as paquete:
    paquete.write(SALIDA,arcname="eduai_voz_kaggle.wav")
print("LISTO: descarga",ZIP.name,"desde Output → /kaggle/working")
display(FileLink(str(ZIP)))
print("En EDUAI selecciona tu perfil y usa Importar ZIP o WAV de Kaggle.")